# Actividad 2.8 — Laboratorio: Implementación de Regresión Lineal
**IEI097 Minería de Datos · Unidad II** · Trabajo individual
**Estudiante:** Miguel Angel Chura Atora

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xXElGrisXx/1.9/blob/main/2.8/Actividad_2.8_Regresion_Lineal.ipynb)

**Objetivo.** Entrenar regresiones lineales con scikit-learn, evaluarlas con partición entrenamiento/prueba (R², MAE, RMSE), interpretar coeficientes y residuos, y justificar decisiones frente a *data leakage*, multicolinealidad y heterogeneidad.

| Bloque | Tema | Dataset |
|---|---|---|
| 1 | Regresión simple | Breast Cancer Wisconsin (`uciml/breast-cancer-wisconsin-data`) |
| 2 | Regresión múltiple | FIFA 19 (`karangadiya/fifa19`) |
| 3 | Transferencia | Dataset propio de Kaggle (Laboratorio 01) |

**Pregunta guía de toda la sesión:** no «¿qué tan alto es mi R²?», sino «¿puedo confiar en este modelo y explicar por qué?».

## Configuración

In [ ]:
!pip -q install kagglehub statsmodels

In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
SEED = 42   # semilla fija para que la partición sea reproducible

**Por qué esta configuración.**
- Se instala `statsmodels` porque ahí está el cálculo del VIF (multicolinealidad), y `kagglehub` para descargar los datasets sin subir archivos a mano.
- Se fija `SEED = 42` para que la partición entrenamiento/prueba sea siempre la misma: los resultados son reproducibles y cualquiera que ejecute el notebook obtiene las mismas métricas.

Las credenciales de Kaggle se leen desde **Colab Secrets** (`KAGGLE_USERNAME` y `KAGGLE_KEY`) y nunca se escriben ni se imprimen en el notebook. **Por qué:** este notebook se publica en GitHub; una clave escrita en una celda quedaría pública para cualquiera.

In [ ]:
# Credenciales desde Colab Secrets (no se imprimen)
try:
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    print("Credenciales de Kaggle cargadas desde Colab Secrets.")
except Exception as e:
    print("No se pudieron leer los Secrets:", type(e).__name__)
    print("Crea KAGGLE_USERNAME y KAGGLE_KEY en el panel de llaves de Colab y activa el acceso del notebook.")

### Funciones auxiliares (se reutilizan en los tres bloques)
**Por qué funciones.** Los tres bloques repiten el mismo flujo (EDA, VIF, métricas, residuos). Definirlo una sola vez evita errores de copiar y pegar y asegura que los tres bloques se evalúen con el mismo criterio.

**Por qué tres métricas.** Cada una responde una pregunta distinta y juntas evitan conclusiones engañosas:
- **R²:** qué proporción de la variación del objetivo explica el modelo (sin unidades, de 0 a 1). Sirve para comparar modelos, pero no dice cuánto se equivoca ni si la forma del modelo es correcta.
- **MAE (error absoluto medio):** cuánto se equivoca el modelo en promedio, en las mismas unidades de la variable; es fácil de explicar a quien no conoce estadística.
- **RMSE (raíz del error cuadrático medio):** también en las unidades de la variable, pero castiga más los errores grandes. Si el RMSE es bastante mayor que el MAE, hay errores grandes ocasionales.

In [ ]:
import kagglehub

def descargar_csv(handle, preferido=None):
    """Descarga un dataset de Kaggle, lista sus CSV y devuelve la ruta del principal."""
    path = kagglehub.dataset_download(handle)
    csvs = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)
    print("Archivos CSV:", [os.path.relpath(p, path) for p in csvs])
    if preferido:
        elegido = next(p for p in csvs if os.path.basename(p).lower() == preferido.lower())
    else:
        elegido = max(csvs, key=os.path.getsize)   # el más grande suele ser el principal
    print("CSV principal:", os.path.basename(elegido))
    return elegido

def metricas(y_real, y_pred):
    return {"R²": r2_score(y_real, y_pred),
            "MAE": mean_absolute_error(y_real, y_pred),
            "RMSE": float(np.sqrt(mean_squared_error(y_real, y_pred)))}

def eda_univariado(df, cols, titulo):
    """Histograma (con densidad) y box-plot de cada variable."""
    fig, ax = plt.subplots(2, len(cols), figsize=(4.2 * len(cols), 7), squeeze=False)
    for j, c in enumerate(cols):
        sns.histplot(df[c].dropna(), kde=True, ax=ax[0][j]); ax[0][j].set_title(f"Distribución de {c}")
        sns.boxplot(x=df[c].dropna(), ax=ax[1][j]); ax[1][j].set_title(f"Box-plot de {c}")
    fig.suptitle(titulo, y=1.01); plt.tight_layout(); plt.show()

def mapa_correlacion(df, cols, titulo):
    corr = df[cols].corr()
    lado = max(6, 0.62 * len(cols) + 2)
    plt.figure(figsize=(lado, lado * 0.85))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, linewidths=.3,
                annot_kws={"size": 8})
    plt.title(titulo); plt.tight_layout(); plt.show()
    return corr

def vif_tabla(X):
    Xc = sm.add_constant(X.astype(float))
    v = [variance_inflation_factor(Xc.values, i) for i in range(1, Xc.shape[1])]
    return pd.DataFrame({"variable": X.columns, "VIF": v}).sort_values("VIF", ascending=False).reset_index(drop=True)

def reducir_por_vif(X, umbral=10.0):
    """Quita, de a una, la variable con mayor VIF hasta que todas queden bajo el umbral."""
    X = X.copy(); pasos = []
    while X.shape[1] > 2:
        t = vif_tabla(X)
        if t["VIF"].max() <= umbral:
            break
        peor = t.iloc[0]
        pasos.append((peor["variable"], round(float(peor["VIF"]), 1)))
        X = X.drop(columns=peor["variable"])
    return X, pasos

def graf_residuos(ax, y_pred, resid, titulo):
    ax.scatter(y_pred, resid, alpha=.6, s=18)
    ax.axhline(0, color="red", lw=1)
    ax.set(title=titulo, xlabel="Valor predicho", ylabel="Residuo (real − predicho)")

---
# Bloque 1 · Regresión lineal simple con Breast Cancer
**Datos:** 569 filas de mediciones de núcleos celulares. **Modelos:** `perimeter_mean ~ radius_mean` y `area_mean ~ radius_mean`.

### 1.1 Carga e inspección

In [ ]:
csv_bc = descargar_csv("uciml/breast-cancer-wisconsin-data", preferido="data.csv")
bc_raw = pd.read_csv(csv_bc)
print("Filas x columnas:", bc_raw.shape)

In [ ]:
bc_raw.info()
print("\nValores nulos por columna (solo las que tienen):")
nulos = bc_raw.isna().sum(); print(nulos[nulos > 0])
bc_raw.head()

**Columnas que no son mediciones y se descartan:**
- `id`: identificador del paciente; no describe el núcleo celular y solo agregaría ruido.
- `diagnosis`: etiqueta categórica (benigno/maligno), no una medición. Se conserva aparte solo para colorear gráficos.
- `Unnamed: 32`: columna completamente vacía (todos nulos), producto de una coma sobrante en el CSV.

In [ ]:
no_medicion = [c for c in ["id", "diagnosis", "Unnamed: 32"] if c in bc_raw.columns]
diagnostico = bc_raw["diagnosis"] if "diagnosis" in bc_raw.columns else None
bc = bc_raw.drop(columns=no_medicion)
print("Columnas descartadas:", no_medicion)
print("Forma del dataset de modelado:", bc.shape, "| nulos restantes:", int(bc.isna().sum().sum()))

**Por qué se inspecciona antes de modelar.** Ver dimensiones, tipos de dato y nulos permite detectar a tiempo columnas que no sirven (identificadores, etiquetas, columnas vacías). Un modelo entrenado con columnas inadecuadas puede mostrar métricas que parecen buenas sin tener sentido real.

### 1.2 EDA del ejercicio

In [ ]:
VARS_B1 = ["radius_mean", "perimeter_mean", "area_mean"]
eda_univariado(bc, VARS_B1, "Bloque 1 · variables a modelar")

In [ ]:
cols_mean = [c for c in bc.columns if c.endswith("_mean")]
corr_bc = mapa_correlacion(bc, cols_mean, "Bloque 1 · correlaciones entre las 10 mediciones promedio")
print(corr_bc.loc["radius_mean", ["perimeter_mean", "area_mean"]].round(3))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a, y in zip(ax, ["perimeter_mean", "area_mean"]):
    sns.regplot(data=bc, x="radius_mean", y=y, ax=a, scatter_kws={"alpha": .5, "s": 18}, line_kws={"color": "red"})
    a.set_title(f"{y} vs radius_mean (con línea de regresión)")
plt.tight_layout(); plt.show()

**Qué muestra el EDA.** `radius_mean` se correlaciona casi perfectamente con `perimeter_mean` (r ≈ 0,998) y con `area_mean` (r ≈ 0,987); por eso el radio es el predictor natural para ambos modelos. Las tres variables tienen distribución sesgada a la derecha y valores atípicos altos en el box-plot (tumores grandes), sobre todo `area_mean`. Geométricamente el perímetro crece con el radio y el área con su cuadrado, lo que anticipa que la recta se ajustará mejor al perímetro que al área.

**Por qué estos gráficos.**
- **Histograma y box-plot:** muestran la forma de la distribución y los valores atípicos. Importan porque la regresión lineal minimiza errores al cuadrado, y un valor extremo puede tirar de la recta.
- **Mapa de correlaciones:** permite elegir predictores con fundamento (los más asociados al objetivo) y detectar variables redundantes entre sí.
- **Gráfico de regresión con su línea:** deja ver a simple vista si la relación es realmente una recta antes de confiar en una métrica.

### 1.3 Los dos modelos, partición 80/20 y métricas

In [ ]:
def ajustar_simple(df, x, y):
    Xtr, Xte, ytr, yte = train_test_split(df[[x]], df[y], test_size=0.2, random_state=SEED)
    m = LinearRegression().fit(Xtr, ytr)
    pred = m.predict(Xte)
    return m, Xte, yte, pred

modelos_b1, filas = {}, []
for y in ["perimeter_mean", "area_mean"]:
    m, Xte, yte, pred = ajustar_simple(bc, "radius_mean", y)
    modelos_b1[y] = (m, Xte, yte, pred)
    filas.append({"modelo": f"{y} ~ radius_mean", "pendiente": m.coef_[0], "intercepto": m.intercept_,
                  **{f"{k} (prueba)": v for k, v in metricas(yte, pred).items()}})
tabla_b1 = pd.DataFrame(filas).set_index("modelo")
tabla_b1.round(3)

In [ ]:
for y, (m, *_ ) in modelos_b1.items():
    print(f"{y}  =  {m.coef_[0]:.3f} · radius_mean  {m.intercept_:+.3f}")

**Interpretación de pendiente e intercepto (con mis palabras).**
- **Perímetro:** la pendiente (≈ 6,86) dice que por cada unidad más de radio el perímetro aumenta en promedio unas 6,9 unidades. Es coherente con la geometría (perímetro = 2π·radio ≈ 6,28·radio), así que el modelo captura una relación real. El intercepto (≈ −5) es el perímetro que predice para radio 0, que no tiene significado físico: solo ancla la recta, porque el rango observado del radio parte en ≈ 7.
- **Área:** la pendiente (≈ 99) indica que cada unidad más de radio suma unas 99 unidades de área *en promedio sobre el rango observado*; como el área crece con el radio al cuadrado, ese incremento no es constante y la pendiente es solo un promedio. El intercepto (≈ −744) es un valor negativo imposible para un área, señal de que una recta no es la forma correcta de esta relación.

**Por qué esta forma de evaluar.**
- **Partición 80/20:** el modelo se ajusta con el 80 % de los datos y se evalúa con el 20 % que nunca vio. Medir el error con los mismos datos de entrenamiento daría una imagen demasiado optimista, porque un modelo siempre se ajusta mejor a lo que ya conoce.
- **`random_state` fijo:** hace que la partición sea reproducible.
- **`LinearRegression`:** encuentra la recta que minimiza la suma de los errores al cuadrado (mínimos cuadrados); es el método estándar y sus coeficientes se interpretan directamente.
- **Mismo predictor y misma partición en ambos modelos:** así la comparación entre perímetro y área es justa; la diferencia viene del objetivo, no de la partición.

### 1.4 Gráficos de la regresión y de residuos

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(12, 9))
for i, (y, (m, Xte, yte, pred)) in enumerate(modelos_b1.items()):
    xs = np.linspace(bc["radius_mean"].min(), bc["radius_mean"].max(), 100).reshape(-1, 1)
    ax[i][0].scatter(bc["radius_mean"], bc[y], alpha=.45, s=16, label="datos")
    ax[i][0].plot(xs, m.predict(pd.DataFrame(xs, columns=["radius_mean"])), color="red", label="recta ajustada")
    ax[i][0].set(title=f"Regresión: {y} ~ radius_mean", xlabel="radius_mean", ylabel=y); ax[i][0].legend()
    graf_residuos(ax[i][1], pred, yte - pred, f"Residuos (prueba): {y}")
plt.tight_layout(); plt.show()

**Por qué se grafican los residuos.** El R² resume el ajuste en un solo número y puede ocultar problemas. El residuo (real − predicho) debería verse como ruido repartido al azar alrededor de cero; si dibuja una forma (curva, embudo), el modelo comete un error sistemático que ninguna métrica global muestra.

### Pregunta guía: ¿un R² alto es suficiente para concluir que una recta es el modelo adecuado?

**No.** Los dos modelos tienen un R² altísimo en prueba (≈ 0,996 el del perímetro y ≈ 0,981 el del área), pero los gráficos de residuos cuentan historias distintas:
- En el **perímetro** los residuos se reparten sin una forma clara alrededor de cero (con un valor atípico aislado y una dispersión que crece algo en los valores altos): una recta es un modelo adecuado.
- En el **área** los residuos dibujan una **curva en forma de U**: el modelo subestima el área en los extremos (residuos positivos) y la sobreestima en el centro (residuos negativos); para radios pequeños llega incluso a predecir áreas negativas, algo imposible. El patrón sistemático indica que la relación es curva (área ∝ radio²), de modo que el R² alto solo mide qué tanto de la varianza se explica, no si la forma funcional es correcta. Para el área convendría usar `radius_mean²` como predictor o transformar la variable.

Un R² alto con residuos que tienen estructura significa que el modelo parece bueno pero comete errores sistemáticos, y se detecta únicamente mirando los residuos.

---
# Bloque 2 · Regresión lineal múltiple con FIFA 19
**Variable objetivo:** `Overall` (valoración general del jugador).

### 2.1 Carga

In [ ]:
csv_fifa = descargar_csv("karangadiya/fifa19", preferido="data.csv")
fifa = pd.read_csv(csv_fifa)
print("Filas x columnas:", fifa.shape)
print("¿Existe la variable objetivo 'Overall'?", "Overall" in fifa.columns)
fifa.head(3)

### 2.2 EDA del ejercicio

In [ ]:
num = fifa.select_dtypes(include="number")
EXCLUIR = [c for c in ["Unnamed: 0", "ID", "Jersey Number", "Overall"] if c in num.columns]
corr_overall = num.drop(columns=EXCLUIR).corrwith(fifa["Overall"]).sort_values(key=abs, ascending=False)

top15 = corr_overall.head(15)
ax = top15[::-1].plot.barh(figsize=(7, 6), color="#3b6ea5")
ax.set_title("Correlación con Overall (15 variables numéricas más correlacionadas)")
ax.set_xlabel("correlación de Pearson"); plt.tight_layout(); plt.show()
print(top15.round(3))

In [ ]:
eda_univariado(fifa, ["Overall"] + list(top15.index[:3]), "Bloque 2 · variable objetivo y 3 candidatas más correlacionadas")

In [ ]:
cols_corr = ["Overall"] + list(top15.index)
corr_f = mapa_correlacion(fifa, cols_corr, "Bloque 2 · correlaciones de las candidatas con Overall")

**Qué muestra el EDA.** `Overall` tiene una distribución aproximadamente simétrica y centrada en torno a 66. El gráfico de barras ordena las variables numéricas por su correlación con `Overall` y sirve para decidir con cuáles trabajar; el mapa de calor muestra además que varias candidatas están muy correlacionadas **entre sí** (las habilidades con balón), lo que se revisa en la sección de multicolinealidad.

**Por qué el EDA se hace sobre el objetivo y las candidatas.** `Overall` es lo que se quiere explicar: su distribución indica si la regresión lineal es razonable (una variable muy sesgada o con muchos atípicos complica el ajuste). Ordenar las variables por correlación es un filtro rápido para no probar a ciegas entre casi 90 columnas, pero tiene dos límites: mide solo relación lineal y no distingue entre una variable legítima y otra que ya contiene la respuesta. Por eso a continuación se aplica el control de fuga de datos.

### 2.3 Selección de predictores y control de *data leakage*

Para cada candidata se aplica la pregunta de control: *¿existiría esta variable antes de conocer el valor que quiero predecir, o se calcula a partir de él o de la misma fuente?*

**Variables que descarto, y por qué** (las que existen en el dataset):

In [ ]:
DESCARTADAS = {
    "Potential": "Es la estimación del Overall máximo que alcanzará el jugador; sale del mismo modelo de valoración y en los jugadores jóvenes casi coincide con Overall. Fuga por fuente común.",
    "Special": "Es la suma de los atributos individuales: una variable derivada que ya contiene la información de las demás y no es una medición independiente.",
    "International Reputation": "La reputación es en buena parte una consecuencia de ser un jugador de Overall alto; no existiría antes de conocer la valoración.",
    "Value": "El valor de mercado es una consecuencia del Overall (el mercado paga por la valoración). Además viene como texto.",
    "Wage": "El sueldo es consecuencia del nivel del jugador. Además viene como texto.",
    "Release Clause": "La cláusula de rescisión se calcula a partir del valor del jugador; es una consecuencia del Overall.",
}
descartadas = {k: v for k, v in DESCARTADAS.items() if k in fifa.columns}
pd.DataFrame({"variable descartada": list(descartadas), "motivo": list(descartadas.values())})

In [ ]:
candidatas = [c for c in top15.index if c not in descartadas]
print("Candidatas finales (numéricas, sin fuga):", candidatas)
assert len(candidatas) >= 8, "Se piden al menos 8 candidatas: amplía top15 a más variables."

**Por qué importa la fuga de datos.** Si se deja una variable que ya contiene la respuesta, el modelo muestra un R² casi perfecto en el laboratorio, pero no sirve para predecir un jugador nuevo, cuya valoración aún no se conoce. Por eso las variables se descartan **antes** de entrenar y por su significado, no después de ver cuál sube el R²: elegir variables mirando la nota sería ajustar el modelo a la prueba.

### 2.4 Multicolinealidad: correlación y VIF

In [ ]:
datos = fifa[["Overall", "Position"] + candidatas].dropna()
print("Filas usadas (sin nulos):", len(datos), "de", len(fifa))
vif_inicial = vif_tabla(datos[candidatas])
vif_inicial.round(2)

In [ ]:
X_vif, pasos = reducir_por_vif(datos[candidatas], umbral=10.0)
print("Variables eliminadas por VIF > 10 (en orden):", pasos if pasos else "ninguna")
print("Variables que se mantienen:", list(X_vif.columns))
vif_tabla(X_vif).round(2)

**Decisión.** Elimino de a una la variable con mayor VIF hasta que todas quedan bajo 10 (regla práctica del enunciado: 5 a 10 es alerta). Se mantienen las variables de la última tabla de arriba; las eliminadas aparecen en el listado paso a paso.

**Por qué VIF y no solo la matriz de correlación.** La correlación compara variables de a pares; el VIF detecta cuándo una variable se puede reconstruir a partir de **varias** otras a la vez.
**Por qué umbral 10.** Es la regla práctica habitual: un VIF alto indica que el coeficiente de esa variable es inestable.
**Por qué de a una.** Al sacar una variable cambia el VIF de las demás (a veces el problema desaparece solo); sacar varias de golpe podría eliminar variables útiles.
**Para qué.** Para que los coeficientes sean estables e interpretables: con predictores redundantes (control de balón, regate y pase corto miden casi lo mismo) los coeficientes «se pelean» entre sí y pueden cambiar de signo.

### 2.5 Modelo, métricas en entrenamiento y prueba, y coeficientes

In [ ]:
predictores = list(X_vif.columns)
Xtr, Xte, ytr, yte = train_test_split(datos[predictores], datos["Overall"], test_size=0.2, random_state=SEED)
modelo = LinearRegression().fit(Xtr, ytr)
pred_tr, pred_te = modelo.predict(Xtr), modelo.predict(Xte)
met = pd.DataFrame({"entrenamiento": metricas(ytr, pred_tr), "prueba": metricas(yte, pred_te)}).T
print("Predictores:", predictores)
met.round(3)

In [ ]:
coefs = pd.DataFrame({"coeficiente": modelo.coef_}, index=predictores)
coefs["desv. estándar de X"] = Xtr.std()
coefs["efecto de +1 desv. estándar"] = coefs["coeficiente"] * coefs["desv. estándar de X"]
coefs = coefs.sort_values("efecto de +1 desv. estándar", key=abs, ascending=False)
print("Intercepto:", round(modelo.intercept_, 3))
display(coefs.round(3))
print()
for v in coefs.index[:3]:
    unidad = "año" if v == "Age" else "punto"
    print(f"• {v}: manteniendo constantes las demás variables, +1 {unidad} de {v} cambia el Overall en "
          f"{coefs.loc[v, 'coeficiente']:+.3f} puntos.")

In [ ]:
dif = met.loc["entrenamiento", "R²"] - met.loc["prueba", "R²"]
print(f"R² entrenamiento = {met.loc['entrenamiento','R²']:.3f} | R² prueba = {met.loc['prueba','R²']:.3f} | diferencia = {dif:+.3f}")
print("Lectura:", "las métricas son parecidas, no hay señal de sobreajuste." if abs(dif) < 0.03
      else "el modelo rinde claramente mejor en entrenamiento que en prueba: hay señal de sobreajuste.")
print(f"RMSE prueba = {met.loc['prueba','RMSE']:.2f} puntos de Overall: un error típico de ese tamaño al predecir un jugador nuevo.")

mejor = corr_overall[predictores].abs().idxmax()
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
muestra = datos.sample(min(3000, len(datos)), random_state=SEED)
sns.regplot(data=muestra, x=mejor, y="Overall", ax=ax[0], scatter_kws={"alpha": .3, "s": 10}, line_kws={"color": "red"})
ax[0].set_title(f"Predictor más correlacionado con Overall: {mejor} (con su línea de regresión)")
ax[1].scatter(yte, pred_te, alpha=.3, s=10)
lim = [min(yte.min(), pred_te.min()), max(yte.max(), pred_te.max())]
ax[1].plot(lim, lim, color="red", lw=1); ax[1].set(title="Valores reales vs. predichos (prueba)", xlabel="Overall real", ylabel="Overall predicho")
plt.tight_layout(); plt.show()

**Por qué se reportan métricas en entrenamiento y en prueba.** Comparar ambas permite detectar sobreajuste: si el modelo es muy bueno en entrenamiento y mucho peor en prueba, memorizó en vez de aprender. Si son parecidas, el modelo generaliza.

**Por qué el efecto de +1 desviación estándar.** Los coeficientes están en unidades distintas (puntos de atributo, años de edad), así que no se pueden comparar directamente. Multiplicar cada coeficiente por la desviación estándar de su variable pone todos en la misma escala (cuántos puntos de Overall cambia una variación típica de esa variable).

**Por qué se dice “manteniendo constantes las demás variables”.** En la regresión múltiple, cada coeficiente mide el efecto de esa variable **una vez descontado el efecto de las otras**; no es el efecto que se vería mirando la variable sola. Es una asociación, no una prueba de causalidad: mejorar un atributo de un jugador no garantiza que su Overall suba en la misma proporción.

**Cómo leer los tres coeficientes de mayor efecto.** Quedan impresos arriba con su unidad: los atributos se miden en puntos (escala 0 a 99) y `Age` en años. Un coeficiente positivo y grande en `Reactions` significa que, a igualdad de las demás variables, cada punto extra de ese atributo se asocia a más Overall.

### 2.6 Heterogeneidad: métricas por subgrupo de posición

In [ ]:
def grupo_posicion(p):
    if p == "GK": return "Arquero"
    if p in {"CB", "LCB", "RCB", "LB", "RB", "LWB", "RWB"}: return "Defensa"
    if p in {"CM", "CDM", "CAM", "LM", "RM", "LCM", "RCM", "LDM", "RDM", "LAM", "RAM"}: return "Mediocampo"
    return "Ataque"

pos_te = datos.loc[Xte.index, "Position"].map(grupo_posicion)
res_te = yte - pred_te
filas = []
for g in sorted(pos_te.unique()):
    idx = pos_te == g
    filas.append({"grupo": g, "n (prueba)": int(idx.sum()), "sesgo medio (residuo)": float(res_te[idx].mean()),
                  **metricas(yte[idx], pred_te[idx])})
het = pd.DataFrame(filas).set_index("grupo")
display(het.round(3))
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
sns.boxplot(x=pos_te, y=res_te, ax=ax[0]); ax[0].axhline(0, color="red", lw=1)
ax[0].set(title="Residuos por grupo de posición", xlabel="", ylabel="Residuo (real − predicho)")
for g in sorted(pos_te.unique()):
    ax[1].scatter(pred_te[(pos_te == g).values], res_te[pos_te == g], s=8, alpha=.4, label=g)
ax[1].axhline(0, color="red", lw=1); ax[1].legend(); ax[1].set(title="Residuos vs. predichos, por grupo", xlabel="Valor predicho")
plt.tight_layout(); plt.show()
peor = het["MAE"].idxmax(); mejor_g = het["MAE"].idxmin()
print(f"Mayor error: {peor} (MAE {het.loc[peor,'MAE']:.2f}); menor error: {mejor_g} (MAE {het.loc[mejor_g,'MAE']:.2f}).")
sesgo = het.loc[peor, "sesgo medio (residuo)"]
print(f"En {peor} el error es {het.loc[peor,'MAE']/het.loc[mejor_g,'MAE']:.1f} veces el de {mejor_g}, y el sesgo medio es {sesgo:+.2f}: "
      + ("el modelo sobreestima a ese grupo (predice más de lo real)." if sesgo < 0 else "el modelo subestima a ese grupo (predice menos de lo real)."))

**Por qué se calculan las métricas por subgrupo.** Un error global promedia a todos los jugadores: puede verse bien aunque el modelo falle de forma sistemática en un grupo pequeño. Separar por posición permite ver esa falla.

**Comentario comparativo.** Un único modelo global mezcla comportamientos distintos. Los arqueros se valoran con atributos propios (`GKDiving`, `GKReflexes`, etc.) que no están entre los predictores, mientras que los jugadores de campo se valoran con habilidades con balón. Por eso se espera que el error y el sesgo (residuo medio distinto de cero) cambien entre grupos; la tabla, el box-plot y la frase impresa arriba muestran en qué grupo falla el modelo y en qué dirección. **Qué haría:** entrenar un modelo por grupo (como mínimo separar a los arqueros) o incorporar como predictores los atributos específicos de cada posición. **Por qué importa:** si se usara este modelo para valorar arqueros, se equivocaría de forma sistemática aunque el error global parezca aceptable.

### 2.7 Desafío opcional: predecir `Value`

**Por qué convertir `Value` a número.** Viene como texto («€110.5M», «€565K»); la regresión necesita números, así que se transforma a euros.
**Por qué mirar su distribución antes de modelar.** El valor de mercado está muy sesgado a la derecha (pocos jugadores valen decenas de millones y la mayoría muy poco), y una recta ajustada a esos datos se ve dominada por los valores extremos y puede predecir valores negativos.
**Por qué probar `log(1 + Value)`.** El logaritmo hace la distribución más simétrica y convierte efectos multiplicativos (cada punto de Overall multiplica el valor) en efectos aditivos, que sí son lineales. Para comparar con justicia se mide el error en euros en ambos casos, no el R², porque el R² en escala logarítmica no es comparable con el de la escala original.

In [ ]:
def a_euros(s):
    if pd.isna(s): return np.nan
    s = str(s).replace("€", "").strip(); mult = 1.0
    if s.endswith("M"): mult, s = 1e6, s[:-1]
    elif s.endswith("K"): mult, s = 1e3, s[:-1]
    try: return float(s) * mult
    except ValueError: return np.nan

try:
    val = fifa["Value"].map(a_euros).rename("Value_eur")
    d2 = pd.concat([val, fifa[predictores]], axis=1).dropna()
    d2 = d2[d2["Value_eur"] > 0]
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
    sns.histplot(d2["Value_eur"], bins=60, ax=ax[0]); ax[0].set_title("Value (€) — muy sesgada a la derecha")
    sns.histplot(np.log1p(d2["Value_eur"]), bins=60, ax=ax[1]); ax[1].set_title("log(1 + Value) — más simétrica")
    plt.tight_layout(); plt.show()
    Xa, Xb, ya, yb = train_test_split(d2[predictores], d2["Value_eur"], test_size=0.2, random_state=SEED)
    m_raw = LinearRegression().fit(Xa, ya)
    m_log = LinearRegression().fit(Xa, np.log1p(ya))
    p_raw = m_raw.predict(Xb); p_log = m_log.predict(Xb)
    print("Sin transformar    → R² prueba:", round(r2_score(yb, p_raw), 3), "| MAE (€):", f"{mean_absolute_error(yb, p_raw):,.0f}",
          "| predicciones negativas:", int((p_raw < 0).sum()))
    print("Objetivo log(1+V)  → R² prueba (escala log):", round(r2_score(np.log1p(yb), p_log), 3),
          "| MAE (€, vuelto a euros):", f"{mean_absolute_error(yb, np.expm1(p_log)):,.0f}")
    print("Mediana de Value (€):", f"{yb.median():,.0f}", "→ compara los MAE con este valor típico.")
except Exception as e:
    print("No se pudo completar el desafío opcional:", type(e).__name__, e)

---
# Bloque 3 · Transferencia a mi propio dataset
Flujo completo sobre el dataset de Kaggle del Laboratorio 01.

### 3.1 Configuración del dataset
**Dataset:** *Walmart Dataset* (`yasserh/walmart-dataset`): ventas semanales (`Weekly_Sales`, en dólares) de 45 tiendas, con `Temperature`, `Fuel_Price`, `CPI`, `Unemployment` y `Holiday_Flag` (1 si la semana incluye un feriado).
**Variable objetivo:** `Weekly_Sales` (numérica continua). **Segmentación para heterogeneidad:** `tamano_tienda` (tiendas pequeñas, medianas y grandes, según su venta semanal promedio).

**Por qué este dataset.** Es un caso de ventas con una variable objetivo continua y varios predictores numéricos que se conocen antes de la semana de venta (clima, combustible, IPC, desempleo, feriados), de modo que no hay fuga evidente.
**Por qué `Store` se descarta.** Es un código identificador (1 a 45), no una magnitud: la regresión lo trataría como si la tienda 40 fuera «más» que la tienda 4, lo que no tiene sentido.
**Por qué `tamano_tienda` no es un predictor.** Se construye con las ventas promedio de cada tienda, es decir, con el propio objetivo; usarlo para predecir sería fuga de datos. Se usa **solo** para comparar grupos en la sección de heterogeneidad.

In [ ]:
# Configuración del Bloque 3 (cámbiala si usas otro dataset)
DATASET_HANDLE = "yasserh/walmart-dataset"
ARCHIVO_CSV = None              # None = elige el CSV más grande (aquí hay uno solo)
TARGET = "Weekly_Sales"         # variable numérica continua a predecir
SEGMENTO = "tamano_tienda"      # columna para revisar heterogeneidad (se crea abajo)
DESCARTADAS_3 = {"Store": "Es un identificador (1 a 45), no una magnitud: su número no significa «más» o «menos» de nada."}
# Unidades, para escribir las interpretaciones con sentido:
UNIDADES = {"Temperature": "°F", "Fuel_Price": "dólares por galón", "CPI": "puntos del índice de precios",
            "Unemployment": "puntos porcentuales", "Holiday_Flag": "semana con feriado (1) frente a semana sin feriado (0)"}
UNIDAD_TARGET = "dólares por semana"

csv_ds = descargar_csv(DATASET_HANDLE, preferido=ARCHIVO_CSV)
ds = pd.read_csv(csv_ds)
# Segmento para comparar grupos: tamaño de tienda según su venta semanal promedio (NO es un predictor)
ds["tamano_tienda"] = pd.qcut(ds.groupby("Store")[TARGET].transform("mean"), 3,
                              labels=["pequeña", "mediana", "grande"]).astype(str)
print("Filas x columnas:", ds.shape)

In [ ]:
# Variables de texto con solo 2 valores (yes/no, male/female) → 0/1, para poder usarlas como predictores
for c in [c for c in ds.select_dtypes(exclude="number").columns if ds[c].nunique() == 2]:
    v0, v1 = sorted(ds[c].dropna().unique())
    ds[c + "_bin"] = (ds[c] == v1).astype(int)
    print(f"{c}_bin: 1 = '{v1}', 0 = '{v0}'")
print("Nulos totales:", int(ds.isna().sum().sum()))
ds.info()
ds.head()

### 3.2 EDA propio

In [ ]:
num3 = ds.select_dtypes(include="number")
corr3 = num3.drop(columns=[TARGET]).corrwith(ds[TARGET]).sort_values(key=abs, ascending=False)
top3 = corr3.head(12)
ax = top3[::-1].plot.barh(figsize=(7, 5), color="#3b6ea5"); ax.set_title(f"Correlación con {TARGET}")
plt.tight_layout(); plt.show()
eda_univariado(ds, [TARGET] + list(top3.index[:3]), "Bloque 3 · variable objetivo y 3 candidatas")
mapa_correlacion(ds, [TARGET] + list(top3.index), "Bloque 3 · correlaciones")
mejor3 = top3.index[0]
plt.figure(figsize=(6, 4)); sns.regplot(data=ds, x=mejor3, y=TARGET, scatter_kws={"alpha": .4, "s": 12}, line_kws={"color": "red"})
plt.title(f"{TARGET} ~ {mejor3} (con línea de regresión)"); plt.tight_layout(); plt.show()

**Por qué se repite el EDA en este bloque.** Es el mismo razonamiento de los bloques anteriores aplicado a otro dataset: la distribución y los atípicos del objetivo condicionan el ajuste, y las correlaciones guían la elección de predictores.

### 3.3 Selección de predictores (control de fuga) y multicolinealidad

In [ ]:
descartadas3 = {k: v for k, v in DESCARTADAS_3.items() if k in ds.columns}
print("Descartadas del modelo:", descartadas3 if descartadas3 else "ninguna")
cand3 = [c for c in top3.index if c not in descartadas3]
d3 = ds[[TARGET] + cand3 + ([SEGMENTO] if SEGMENTO and SEGMENTO not in cand3 else [])].dropna()
X3, pasos3 = reducir_por_vif(d3[cand3], umbral=10.0)
print("Eliminadas por VIF > 10:", pasos3 if pasos3 else "ninguna")
vif_tabla(X3).round(2)

**Justificación de predictores.**
- **Qué mantengo y por qué.** Las variables con mayor correlación con las ventas (impresas arriba), porque el EDA muestra que se mueven junto con el objetivo y porque todas se conocen **antes** de la semana de venta (la temperatura, el precio del combustible, el IPC, el desempleo y si la semana tiene feriado no dependen de cuánto se vendió). Por eso no hay fuga de datos.
- **Qué descarto y por qué.** `Store`, porque es un código identificador y no una magnitud (el modelo trataría la tienda 40 como «más» que la tienda 4). Tampoco uso `tamano_tienda` como predictor porque se construye con las ventas promedio, es decir, con el propio objetivo.
- **Multicolinealidad.** Se calculó el VIF de los predictores y se habría eliminado, de a una, la variable más redundante si alguna superaba 10. El resultado impreso arriba indica qué ocurrió. **Por qué se revisa:** con predictores redundantes los coeficientes se vuelven inestables y no se pueden interpretar.

### 3.4 Modelo, métricas y residuos

In [ ]:
pred3_cols = list(X3.columns)
Xtr3, Xte3, ytr3, yte3 = train_test_split(d3[pred3_cols], d3[TARGET], test_size=0.2, random_state=SEED)
m3 = LinearRegression().fit(Xtr3, ytr3)
p_tr3, p_te3 = m3.predict(Xtr3), m3.predict(Xte3)
display(pd.DataFrame({"entrenamiento": metricas(ytr3, p_tr3), "prueba": metricas(yte3, p_te3)}).T.round(3))
coef3 = pd.Series(m3.coef_, index=pred3_cols).sort_values(key=abs, ascending=False)
display(coef3.round(4).to_frame("coeficiente"))
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].scatter(yte3, p_te3, alpha=.5, s=14); l = [yte3.min(), yte3.max()]; ax[0].plot(l, l, color="red", lw=1)
ax[0].set(title="Reales vs. predichos (prueba)", xlabel="real", ylabel="predicho")
graf_residuos(ax[1], p_te3, yte3 - p_te3, "Residuos (prueba)")
plt.tight_layout(); plt.show()

**Por qué se evalúa el modelo en entrenamiento y en prueba, y con residuos.** Igual que en los bloques anteriores: la prueba mide si el modelo generaliza a semanas que no vio, y los residuos muestran errores sistemáticos que una métrica global no revela.

### 3.5 Heterogeneidad

In [ ]:
if SEGMENTO:
    seg = d3.loc[Xte3.index, SEGMENTO]
    if seg.nunique() > 8:
        seg = pd.qcut(seg, 4, duplicates="drop").astype(str)
    r3 = yte3 - p_te3
    filas = [{"grupo": g, "n": int((seg == g).sum()), "sesgo medio": float(r3[seg == g].mean()), **metricas(yte3[seg == g], p_te3[(seg == g).values])}
             for g in sorted(seg.unique()) if (seg == g).sum() >= 5]
    display(pd.DataFrame(filas).set_index("grupo").round(3))
    plt.figure(figsize=(7, 4)); sns.boxplot(x=seg, y=r3); plt.axhline(0, color="red", lw=1)
    plt.title(f"Residuos por grupo de {SEGMENTO}"); plt.tight_layout(); plt.show()
    tabla_h = pd.DataFrame(filas).set_index("grupo")
    pg, mg = tabla_h["MAE"].idxmax(), tabla_h["MAE"].idxmin()
    print(f"El grupo con mayor error es '{pg}' (MAE {tabla_h.loc[pg,'MAE']:,.0f}); el de menor error es '{mg}' (MAE {tabla_h.loc[mg,'MAE']:,.0f}).")
    for g, fila in tabla_h.iterrows():
        print(f"- {g}: sesgo medio {fila['sesgo medio']:+,.0f} -> " + ("el modelo subestima a este grupo (predice menos de lo real)." if fila["sesgo medio"] > 0 else "el modelo sobreestima a este grupo (predice más de lo real)."))
else:
    print("Este dataset no se segmentó: explica el motivo en la celda Markdown de abajo.")

**Comentario de heterogeneidad.** Dividí las tiendas en tres grupos según su venta promedio y calculé el error del modelo en cada uno. Las variables del modelo (clima, combustible, IPC, desempleo, feriado) son iguales para todas las tiendas y no dicen cuál es grande o pequeña; por eso, sin conocer la tienda, el modelo tiende a predecir un valor parecido para todas. El resultado esperable, que la tabla y las frases impresas arriba permiten comprobar, es que las tiendas grandes queden subestimadas (sesgo medio positivo, el real supera al predicho) y las pequeñas sobreestimadas (sesgo medio negativo), con un error mucho mayor en los extremos que en el grupo mediano.
**Por qué importa.** Un error global aceptable puede esconder que el modelo falla de forma sistemática en un tipo de tienda. **Qué mejoraría:** incorporar la tienda como variable categórica (variables indicadoras) o entrenar un modelo por grupo de tienda.

### 3.6 Preguntas de cierre

In [ ]:
from scipy import stats
c0 = coef3.index[0]
u = UNIDADES.get(c0, "unidades de " + c0) if "UNIDADES" in globals() else "unidades de " + c0
ut = UNIDAD_TARGET if "UNIDAD_TARGET" in globals() else "unidades de " + TARGET
print(f"(a) {c0}: manteniendo constantes las demás variables, un aumento de 1 ({u}) se asocia a un cambio de {coef3[c0]:+,.2f} ({ut}) en {TARGET}.")

mt, me = metricas(ytr3, p_tr3), metricas(yte3, p_te3)
rel = (me["RMSE"] / mt["RMSE"] - 1) * 100
print(f"(b) RMSE entrenamiento = {mt['RMSE']:,.0f} | RMSE prueba = {me['RMSE']:,.0f} | diferencia relativa = {rel:+.1f} %")
print("    Lectura:", "el error en prueba es parecido al de entrenamiento: no hay señal de sobreajuste." if abs(rel) < 10
      else "el error en prueba es bastante distinto al de entrenamiento: hay señal de sobreajuste.")
print(f"    R² entrenamiento = {mt['R²']:.3f} | R² prueba = {me['R²']:.3f}")

# (c) diagnóstico objetivo de los supuestos, sobre los residuos de entrenamiento
from statsmodels.stats.diagnostic import het_breuschpagan
res_tr = ytr3 - p_tr3
p_bp = het_breuschpagan(res_tr, sm.add_constant(Xtr3.astype(float)))[1]
p_sh = stats.shapiro(res_tr.sample(min(len(res_tr), 5000), random_state=SEED))[1]
p_cur = float(np.asarray(sm.OLS(np.asarray(res_tr), sm.add_constant(np.c_[p_tr3, p_tr3 ** 2])).fit().pvalues)[2])
print("(c) Diagnóstico de supuestos (nivel de significancia 5 %):")
print(f"    Linealidad (¿hay curvatura en los residuos?): p = {p_cur:.4f} -> " + ("se INCUMPLE (hay curvatura)." if p_cur < 0.05 else "se cumple (no se detecta curvatura)."))
print(f"    Homocedasticidad (varianza constante, Breusch-Pagan): p = {p_bp:.4f} -> " + ("se INCUMPLE (la dispersión de los residuos cambia)." if p_bp < 0.05 else "se cumple."))
print(f"    Normalidad de los residuos (Shapiro-Wilk): p = {p_sh:.4f} -> " + ("se INCUMPLE (residuos no normales)." if p_sh < 0.05 else "se cumple."))

**(a) Interpretación de un coeficiente, con unidades.** La línea (a) impresa arriba da el coeficiente de la variable con mayor peso, con su unidad y la unidad del objetivo. Se lee así: manteniendo constantes las demás variables, un aumento de una unidad de esa variable se asocia a ese cambio en las ventas semanales. **Por qué «manteniendo constantes»:** en la regresión múltiple cada coeficiente descuenta el efecto de las otras variables. Es una asociación, no prueba causalidad: por ejemplo, que una semana con feriado se asocie a más ventas no demuestra que el feriado las cause por sí solo.

**(b) Entrenamiento vs. prueba.** La línea (b) compara el error en ambos conjuntos. **Por qué se hace:** un modelo siempre rinde mejor en los datos con que se entrenó, y lo que importa es si mantiene su desempeño con datos nuevos. Si la diferencia es pequeña, el modelo no memorizó el entrenamiento; con solo cinco predictores y miles de filas el sobreajuste es poco probable. Una diferencia grande indicaría que el modelo no generaliza. Además, un R² bajo en **ambos** conjuntos (si ocurre) no es sobreajuste sino falta de información: estas variables económicas por sí solas explican poco de las ventas.

**(c) Supuesto de la regresión lineal.** Reviso la **homocedasticidad** (que la dispersión de los residuos sea la misma a lo largo de los valores predichos), además de linealidad y normalidad. **Por qué importa:** si la varianza del error cambia, el modelo es menos preciso en unas zonas que en otras y los intervalos de confianza no son confiables. **Evidencia:** el gráfico de residuos vs. predichos de la sección 3.4 (si la nube se ensancha o se estrecha, no hay varianza constante) y la prueba de Breusch-Pagan impresa arriba, cuyo veredicto indica si el modelo cumple o incumple ese supuesto. Un patrón en la heterogeneidad por tamaño de tienda también apunta a que un solo modelo no describe igual de bien a todas las tiendas.

---
## Cierre
Un buen modelo no es el de mayor R², sino el que puedo explicar, defender y reproducir: en el Bloque 1 un R² alto escondía una relación curva, en el Bloque 2 hubo que descartar variables que ya contenían la respuesta y eliminar redundancias antes de interpretar coeficientes, y el error promedio puede ocultar que el modelo falla en un subgrupo.